In [1]:
import pandas as pd
import os
import glob
import ast
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import math
from sklearn.metrics import accuracy_score
from typing import List, Dict, Any

In [5]:
files = os.listdir(r'C:\Users\szymo\Documents\Gomoku\games_history_v2')

In [6]:
def concat_df(path=r'C:\Users\szymo\Documents\Gomoku\games_history_v2'):
    files = os.listdir(path)
    df_final = pd.DataFrame()
    df_list = [pd.read_csv(f"{path}\{file}") for file in files]
    for df in df_list:
        winner = [ast.literal_eval(df['0'][i])['winner'] for i in range(2000)]
        df['winner_player'] = winner
        player_x, player_o = add_players()
        df['player_x'] = player_x
        df['player_o'] = player_o
        df_final = pd.concat([df_final,df])
    return df_final
def add_players(x=1):
    players = ['heuristic_player', 'model_v1_player', 'model_v2_player', 'model_v3_player', 'model_v4_player']
    ordered_pairs = []
    for p1 in players:
        for p2 in players:
            if p1 == p2:
                continue
            ordered_pairs.append((p1, p2))
    player_x = []
    player_o = []
    for j in range(x):
        for p1, p2 in ordered_pairs:
            for i in range(50):
                player_x.append(p1)
                player_o.append(p2)
            for i in range(50):
                player_x.append(p2)
                player_o.append(p1)
    return player_x, player_o

In [7]:
df = concat_df()

In [38]:
df_win_stats = df[['winner_player','player_x','player_o']]

In [42]:
df_win_stats['match'] = df_win_stats['player_x']+" - "+df_win_stats['player_o']

C:\Users\szymo\AppData\Local\Temp\ipykernel_272488\2670851420.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_win_stats['match'] = df_win_stats['player_x']+" - "+df_win_stats['player_o']


In [54]:
raw_stats = df_win_stats.groupby(['match', 'winner_player'])['winner_player'].count() / 26
formatted_df = raw_stats.unstack(level='winner_player')
formatted_df

winner_player,Draw,O,X
match,,,
heuristic_player - model_v1_player,0.153846,27.423077,72.423077
heuristic_player - model_v2_player,0.269231,28.576923,71.153846
heuristic_player - model_v3_player,0.423077,27.461538,72.115385
heuristic_player - model_v4_player,0.038462,23.346154,76.615385
model_v1_player - heuristic_player,0.115385,50.500000,49.384615
model_v1_player - model_v2_player,1.269231,38.769231,59.961538
model_v1_player - model_v3_player,1.038462,35.269231,63.692308
model_v1_player - model_v4_player,NaN,29.384615,70.615385
model_v2_player - heuristic_player,0.384615,48.000000,51.615385


In [31]:
df_win_stats[(df_win_stats['winner_player']=='O')& (df_win_stats['player_o']=='heuristic_player')]

,winner_player,player_x,player_o
50,O,model_v1_player,heuristic_player
51,O,model_v1_player,heuristic_player
52,O,model_v1_player,heuristic_player
53,O,model_v1_player,heuristic_player
54,O,model_v1_player,heuristic_player
...,...,...,...
1642,O,model_v4_player,heuristic_player
1644,O,model_v4_player,heuristic_player
1646,O,model_v4_player,heuristic_player
1648,O,model_v4_player,heuristic_player


In [8]:
def clean_format(df):
    flat_moves = []
    for _, row in df.iterrows():
        for col in df.columns:
            if col in ['Unnamed: 0', 'Unnamed: 0.1']:  
                continue
            cell_value = row[col]
            if pd.notna(cell_value):
                if isinstance(cell_value, str):
                    try:
                        move_dict = ast.literal_eval(cell_value)
                    except:
                        continue
                else:
                    move_dict = cell_value
                flat_moves.append(move_dict)
    df_clean = pd.DataFrame(flat_moves)
    return df_clean

In [9]:
df = clean_format(df)

In [10]:
df

,game_id,move_number,player,move_x,move_y,winner,opening_state
0,5fcb7520-86aa-4825-8c71-43644a8cd514,1,X,8,8,X,"8,8;7,6;6,7;7,7"
1,5fcb7520-86aa-4825-8c71-43644a8cd514,2,O,7,6,X,"8,8;7,6;6,7;7,7"
2,5fcb7520-86aa-4825-8c71-43644a8cd514,3,X,6,7,X,"8,8;7,6;6,7;7,7"
3,5fcb7520-86aa-4825-8c71-43644a8cd514,4,O,7,7,X,"8,8;7,6;6,7;7,7"
4,5fcb7520-86aa-4825-8c71-43644a8cd514,5,X,7,8,X,"8,8;7,6;6,7;7,7"
...,...,...,...,...,...,...,...
2109664,048d8d0c-59d2-42cd-b4c8-fe4da5fd2379,15,X,5,7,X,"7,6;7,8;6,7;8,8"
2109665,048d8d0c-59d2-42cd-b4c8-fe4da5fd2379,16,O,4,7,X,"7,6;7,8;6,7;8,8"
2109666,048d8d0c-59d2-42cd-b4c8-fe4da5fd2379,17,X,7,9,X,"7,6;7,8;6,7;8,8"
2109667,048d8d0c-59d2-42cd-b4c8-fe4da5fd2379,18,O,4,6,X,"7,6;7,8;6,7;8,8"


In [13]:
df.to_csv(r'C:\Users\szymo\Documents\Gomoku\CNN_games_base\all_games_52K.csv')

In [11]:
from gomuku_engine import GomokuEngine
engine = GomokuEngine()

In [4]:
df1 = pd.read_csv(r'C:\Users\szymo\Documents\Gomoku\CNN_games_base\all_games_v1.csv')
df2 = pd.read_csv(r'C:\Users\szymo\Documents\Gomoku\CNN_games_base\all_games_v2.csv')

In [5]:
df = pd.concat([df1,df2])

In [6]:
df.shape

(7113052, 8)

In [7]:
df = df.iloc[:,1:]

In [8]:
df['move_1d'] = df['move_x']*15+df['move_y']

In [9]:
df

,game_id,move_number,player,move_x,move_y,winner,opening_state,move_1d
0,cd728a95-f007-4b04-9b2d-763b1c2432e0,1,X,7,7,O,NaN,112
1,cd728a95-f007-4b04-9b2d-763b1c2432e0,2,O,6,7,O,NaN,97
2,cd728a95-f007-4b04-9b2d-763b1c2432e0,3,X,8,7,O,NaN,127
3,cd728a95-f007-4b04-9b2d-763b1c2432e0,4,O,9,7,O,NaN,142
4,cd728a95-f007-4b04-9b2d-763b1c2432e0,5,X,6,6,O,NaN,96
...,...,...,...,...,...,...,...,...
2109664,048d8d0c-59d2-42cd-b4c8-fe4da5fd2379,15,X,5,7,X,"7,6;7,8;6,7;8,8",82
2109665,048d8d0c-59d2-42cd-b4c8-fe4da5fd2379,16,O,4,7,X,"7,6;7,8;6,7;8,8",67
2109666,048d8d0c-59d2-42cd-b4c8-fe4da5fd2379,17,X,7,9,X,"7,6;7,8;6,7;8,8",114
2109667,048d8d0c-59d2-42cd-b4c8-fe4da5fd2379,18,O,4,6,X,"7,6;7,8;6,7;8,8",66


In [10]:
def check_line_length_np(board_np, row, col, player_val, required_length=5, board_size=15):
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
    for dr, dc in directions:
        count = 1
        r, c = row + dr, col + dc
        while 0 <= r < board_size and 0 <= c < board_size and board_np[r, c] == player_val:
            count += 1
            r += dr
            c += dc
        r, c = row - dr, col - dc
        while 0 <= r < board_size and 0 <= c < board_size and board_np[r, c] == player_val:
            count += 1
            r -= dr
            c -= dc
        if count >= required_length:
            return True
    return False
def evaluate_historical_move(board_np, row, col, player_val, game_winner, moves_to_end, board_size=15):
    weight = 1.0
    opp_val = -player_val  
    board_np[row, col] = player_val
    if check_line_length_np(board_np, row, col, player_val, required_length=5, board_size=board_size):
        board_np[row, col] = 0  
        return 10.0  
    has_my_4 = check_line_length_np(board_np, row, col, player_val, required_length=4, board_size=board_size)
    board_np[row, col] = 0  
    board_np[row, col] = opp_val
    if check_line_length_np(board_np, row, col, opp_val, required_length=5, board_size=board_size):
        weight *= 5.0  
    elif check_line_length_np(board_np, row, col, opp_val, required_length=4, board_size=board_size):
        weight *= 2.5 
    board_np[row, col] = 0  
    if has_my_4 and weight == 1.0:
        weight *= 3.0
    #Neighbours
    r_start, r_end = max(0, row - 1), min(board_size, row + 2)
    c_start, c_end = max(0, col - 1), min(board_size, col + 2)
    slice_3x3 = board_np[r_start:r_end, c_start:c_end]
    my_neighbors = np.sum(slice_3x3 == player_val)
    opp_neighbors = np.sum(slice_3x3 == opp_val)
    neighbor_score = (my_neighbors * 1000) + (opp_neighbors * 700)
    
    if neighbor_score == 0:
        weight *= 0.1
    elif neighbor_score < 1000:
        weight *= 0.5
    # winner moves
    current_player_str = 'X' if player_val == 1 else 'O'
    if current_player_str == game_winner:
        weight *= 1.0
    elif game_winner == 'Draw':
        weight *= 0.75
    else:
        weight *= 0.5
    #final moves
    if moves_to_end <= 2:
        weight *= 2.0
    elif moves_to_end <= 5:
        weight *= 1.5
    elif moves_to_end <= 10:
        weight *= 1.0
    else:
        weight *= 0.6
        
    return weight

def prepare_weighted_data(df, board_size=15, for_deep_learning=False):
    X = []
    y = []
    sample_weights = []
    game_ids = []
    move_numbers = []
    has_opening_state = []
    grouped = df.sort_values(['game_id', 'move_number']).groupby('game_id')
    for game_id, game_moves in grouped:
        winner = game_moves.iloc[0]['winner']
        total_moves = len(game_moves)
        current_board = np.zeros((board_size, board_size), dtype=np.int8)
        contains_opening = pd.notna(game_moves.iloc[0]['opening_state'])
        for move_idx, (_, row) in enumerate(game_moves.iterrows()):
            moves_to_end = total_moves - move_idx - 1
            player_value = 1 if row['player'] == 'X' else -1
            row_idx = int(row['move_x'])
            col_idx = int(row['move_y'])
            weight = evaluate_historical_move(
                board_np=current_board,
                row=row_idx,
                col=col_idx,
                player_val=player_value,
                game_winner=winner,
                moves_to_end=moves_to_end,
                board_size=board_size
            )
            if for_deep_learning:
                my_channel = (current_board == player_value).astype(np.float32)
                opp_channel = (current_board == -player_value).astype(np.float32)
                empty_channel = (current_board == 0).astype(np.float32)
                state = np.stack([my_channel, opp_channel, empty_channel], axis=0)
            else:
                state = current_board.copy().flatten()
            X.append(state)
            y.append(row_idx * board_size + col_idx)  
            sample_weights.append(weight)
            game_ids.append(game_id)
            move_numbers.append(row['move_number'])
            has_opening_state.append(contains_opening)
            current_board[row_idx, col_idx] = player_value
    dtype_X = np.float32 if for_deep_learning else np.int8
    dtype_y = np.int64 if for_deep_learning else np.int32    
    return np.array(X, dtype=dtype_X), np.array(y, dtype=dtype_y), np.array(sample_weights), np.array(game_ids), np.array(move_numbers, dtype=np.int32), np.array(has_opening_state, dtype=bool)

In [38]:
game_ids = df['game_id'].unique()

In [39]:
batch1, batch2, batch3 = (game_ids[:50000],game_ids[50001:100000],game_ids[100001:])

In [42]:
X, Y, weight, game_id, move_number, opening_state = prepare_weighted_data(df[df['game_id'].isin(batch3)], for_deep_learning= True)
np.save(f"b3_X_deep_{current_time}.npy", X)
np.save(f"b3_y_deep_{current_time}.npy", Y)
np.save(f"b3_weights_deep_{current_time}.npy", weight)
np.save(f"b3_game_id_{current_time}.npy", game_id)
np.save(f"b3_move_number_{current_time}.npy", move_number)
np.save(f"b3_opening_state_{current_time}.npy", opening_state)

In [30]:
len(Y)

7113052

In [31]:
len(X)

6905052

In [34]:
len(set(game_id))

163110

In [12]:
from datetime import datetime
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
# Zapisujemy każdą tablicę oddzielnie do natywnego formatu NumPy
np.save(f"X_deep_{current_time}.npy", X)
np.save(f"y_deep_{current_time}.npy", Y)
np.save(f"weights_deep_{current_time}.npy", weight)
np.save(f"game_id_{current_time}.npy", game_id)
np.save(f"move_number_{current_time}.npy", move_number)
np.save(f"opening_state_{current_time}.npy", opening_state)

In [2]:
X = np.load("X_deep_20260707_090401.npy")
Y = np.load("y_deep_20260707_090401.npy")
weight = np.load("weights_deep_20260707_090401.npy")
game_id = np.load("game_id_20260707_090401.npy")

In [8]:
timestamp = "20260707_191353"
X = np.concatenate([np.load(f"CNN_data/b{i}_X_deep_{timestamp}.npy") for i in range(1, 4)], axis=0)
y = np.concatenate([np.load(f"CNN_data/b{i}_y_deep_{timestamp}.npy") for i in range(1, 4)], axis=0)
weights = np.concatenate([np.load(f"CNN_data/b{i}_weights_deep_{timestamp}.npy") for i in range(1, 4)], axis=0)
game_id = np.concatenate([np.load(f"CNN_data/b{i}_game_id_{timestamp}.npy") for i in range(1, 4)], axis=0)
move_number = np.concatenate([np.load(f"CNN_data/b{i}_move_number_{timestamp}.npy") for i in range(1, 4)], axis=0)
opening_state = np.concatenate([np.load(f"CNN_data/b{i}_opening_state_{timestamp}.npy") for i in range(1, 4)], axis=0)

In [46]:
y_train[1]

np.int64(128)

In [9]:
len(X_raw)

7112999

In [21]:
opening_state

array([False, False, False, ..., False, False, False])

In [8]:
sample_weight[0]

np.float64(0.06)

In [10]:
mask = ((opening_state == True) & (move_number > 4)) | (opening_state == False)

In [13]:
X = X[mask]
y = y[mask]
weights = weights[mask]
game_id = game_id[mask]

In [32]:
unique_games = np.unique(game_id)
np.random.seed(42)  
np.random.shuffle(unique_games)
mask_train = np.isin(game_id,unique_games[:round((len(unique_games)*0.9))])
x_train = X[mask_train]
x_val = X[~mask_train]
y_train = y[mask_train]
y_val = y[~mask_train]
weight_train = weights[mask_train]
weight_val = weights[~mask_train]

In [35]:
np.save(f"CNN_data/X_train.npy", x_train)
np.save(f"CNN_data/X_val.npy", x_val)
np.save(f"CNN_data/y_train.npy", y_train)
np.save(f"CNN_data/y_val.npy", y_val)
np.save(f"CNN_data/weight_train.npy", weight_train)
np.save(f"CNN_data/weight_val.npy", weight_val)

In [2]:
x_train = np.load(f"CNN_data/X_train.npy")
x_val = np.load(f"CNN_data/X_val.npy")
y_train = np.load(f"CNN_data/y_train.npy")
y_val = np.load(f"CNN_data/y_val.npy")
weight_train = np.load(f"CNN_data/weight_train.npy")
weight_val = np.load(f"CNN_data/weight_val.npy")

## CNN training

In [3]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

In [4]:
train_dataset = TensorDataset(torch.tensor(x_train), torch.tensor(y_train), torch.tensor(weight_train))
val_dataset = TensorDataset(torch.tensor(x_val), torch.tensor(y_val), torch.tensor(weight_val))

In [5]:
num_cpus = os.cpu_count() or 2
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False, num_workers=0, pin_memory=True)

In [6]:
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch_X, batch_y, batch_w in loader:
        batch_X, batch_y, batch_w = batch_X.to(device, dtype=torch.float32), batch_y.to(device, dtype=torch.long), batch_w.to(device, dtype=torch.float32)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(batch_X)
            loss = (criterion(outputs, batch_y) * batch_w).mean()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * batch_X.size(0)
        correct += outputs.max(1)[1].eq(batch_y).sum().item()
        total += batch_y.size(0)
    return total_loss / total, (correct / total) * 100
def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y, batch_w in loader:
            batch_X, batch_y, batch_w = batch_X.to(device, dtype=torch.float32), batch_y.to(device, dtype=torch.long), batch_w.to(device, dtype=torch.float32)
            with torch.amp.autocast('cuda'):
                outputs = model(batch_X)
                loss = (criterion(outputs, batch_y) * batch_w).mean()
            total_loss += loss.item() * batch_X.size(0)
            correct += outputs.max(1)[1].eq(batch_y).sum().item()
            total += batch_y.size(0)
    return total_loss / total, (correct / total) * 100

### Model 1

In [9]:
model_1 = torch.nn.Sequential(
    torch.nn.Conv2d(3, 64, kernel_size=3, padding=1), torch.nn.ReLU(),
    torch.nn.Conv2d(64, 128, kernel_size=3, padding=1), torch.nn.ReLU(),
    torch.nn.Conv2d(128, 128, kernel_size=3, padding=1), torch.nn.ReLU(),
    torch.nn.Conv2d(128, 64, kernel_size=3, padding=1), torch.nn.ReLU(),
    torch.nn.Flatten(),
    torch.nn.Linear(64 * 15 * 15, 225)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model_1.to(device)
optimizer = optim.Adam(model_1.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(reduction='none')

In [17]:
scaler = torch.amp.GradScaler('cuda')
epochs = 5

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model_1, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_acc = validate_epoch(model_1, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print("-" * 40)

Epoch 1/5
  Train Loss: 1.1866 | Train Acc: 55.10%
  Val Loss:   0.8328 | Val Acc:   59.55%
----------------------------------------
Epoch 2/5
  Train Loss: 0.7794 | Train Acc: 60.63%
  Val Loss:   0.7899 | Val Acc:   60.64%
----------------------------------------
Epoch 3/5
  Train Loss: 0.7329 | Train Acc: 61.77%
  Val Loss:   0.7536 | Val Acc:   61.57%
----------------------------------------
Epoch 4/5
  Train Loss: 0.7073 | Train Acc: 62.41%
  Val Loss:   0.7424 | Val Acc:   61.87%
----------------------------------------
Epoch 5/5
  Train Loss: 0.6909 | Train Acc: 62.83%
  Val Loss:   0.7405 | Val Acc:   62.23%
----------------------------------------


In [18]:
torch.save(model_1.state_dict(), "model_1.pt")

### Model 2

In [26]:
model_2 = torch.nn.Sequential(
    torch.nn.Conv2d(3, 64, kernel_size=5, padding=2), torch.nn.ReLU(),
    torch.nn.Conv2d(64, 128, kernel_size=5, padding=2), torch.nn.ReLU(),
    torch.nn.Conv2d(128, 128, kernel_size=3, padding=1), torch.nn.ReLU(),
    torch.nn.Conv2d(128, 64, kernel_size=3, padding=1), torch.nn.ReLU(),
    torch.nn.Flatten(),
    torch.nn.Linear(64 * 15 * 15, 225)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model_2.to(device)
optimizer = optim.Adam(model_2.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(reduction='none')

In [27]:
scaler = torch.amp.GradScaler('cuda')
epochs = 5

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model_2, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_acc = validate_epoch(model_2, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print("-" * 40)

Epoch 1/5
  Train Loss: 1.2151 | Train Acc: 54.87%
  Val Loss:   0.8060 | Val Acc:   59.75%
----------------------------------------
Epoch 2/5
  Train Loss: 0.7514 | Train Acc: 61.40%
  Val Loss:   0.7464 | Val Acc:   61.65%
----------------------------------------
Epoch 3/5
  Train Loss: 0.6979 | Train Acc: 62.75%
  Val Loss:   0.7248 | Val Acc:   62.48%
----------------------------------------
Epoch 4/5
  Train Loss: 0.6724 | Train Acc: 63.39%
  Val Loss:   0.7161 | Val Acc:   62.84%
----------------------------------------
Epoch 5/5
  Train Loss: 0.6567 | Train Acc: 63.78%
  Val Loss:   0.7324 | Val Acc:   62.84%
----------------------------------------


In [28]:
torch.save(model_2.state_dict(), "model_2.pt")

### Model 3

In [8]:
model_3 = torch.nn.Sequential(
    torch.nn.Conv2d(3, 64, kernel_size=5, padding=2), torch.nn.LeakyReLU(),
    torch.nn.Conv2d(64, 128, kernel_size=5, padding=2), torch.nn.LeakyReLU(),
    torch.nn.Conv2d(128, 128, kernel_size=3, padding=1), torch.nn.LeakyReLU(),
    torch.nn.Conv2d(128, 64, kernel_size=3, padding=1), torch.nn.LeakyReLU(),
    torch.nn.Flatten(),
    torch.nn.Linear(64 * 15 * 15, 225)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model_3.to(device)
optimizer = optim.Adam(model_3.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(reduction='none')

In [9]:
scaler = torch.amp.GradScaler('cuda')
epochs = 5

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model_3, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_acc = validate_epoch(model_3, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print("-" * 40)

Epoch 1/5
  Train Loss: 1.1204 | Train Acc: 56.37%
  Val Loss:   0.7854 | Val Acc:   60.76%
----------------------------------------
Epoch 2/5
  Train Loss: 0.7396 | Train Acc: 61.97%
  Val Loss:   0.7515 | Val Acc:   61.93%
----------------------------------------
Epoch 3/5
  Train Loss: 0.6948 | Train Acc: 63.00%
  Val Loss:   0.7345 | Val Acc:   62.42%
----------------------------------------
Epoch 4/5
  Train Loss: 0.6721 | Train Acc: 63.50%
  Val Loss:   0.7371 | Val Acc:   62.56%
----------------------------------------
Epoch 5/5
  Train Loss: 0.6570 | Train Acc: 63.84%
  Val Loss:   0.7279 | Val Acc:   62.82%
----------------------------------------


In [10]:
torch.save(model_3.state_dict(), "model_3.pt")

### Model 4

In [11]:
model_4 = torch.nn.Sequential(
    torch.nn.Conv2d(3, 64, kernel_size=5, padding=2), torch.nn.GELU(),
    torch.nn.Conv2d(64, 128, kernel_size=5, padding=2), torch.nn.GELU(),
    torch.nn.Conv2d(128, 128, kernel_size=3, padding=1), torch.nn.GELU(),
    torch.nn.Conv2d(128, 64, kernel_size=3, padding=1), torch.nn.GELU(),
    torch.nn.Flatten(),
    torch.nn.Linear(64 * 15 * 15, 225)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model_4.to(device)
optimizer = optim.Adam(model_4.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(reduction='none')

In [12]:
scaler = torch.amp.GradScaler('cuda')
epochs = 5

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model_4, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_acc = validate_epoch(model_4, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print("-" * 40)

Epoch 1/5
  Train Loss: 1.0710 | Train Acc: 56.88%
  Val Loss:   0.8076 | Val Acc:   60.42%
----------------------------------------
Epoch 2/5
  Train Loss: 0.7658 | Train Acc: 61.52%
  Val Loss:   0.7773 | Val Acc:   61.51%
----------------------------------------
Epoch 3/5
  Train Loss: 0.7234 | Train Acc: 62.48%
  Val Loss:   0.7519 | Val Acc:   62.11%
----------------------------------------
Epoch 4/5
  Train Loss: 0.6999 | Train Acc: 62.98%
  Val Loss:   0.7490 | Val Acc:   62.38%
----------------------------------------
Epoch 5/5
  Train Loss: 0.6838 | Train Acc: 63.32%
  Val Loss:   0.7570 | Val Acc:   62.17%
----------------------------------------


In [13]:
torch.save(model_4.state_dict(), "model_4.pt")

## LGBoost

In [20]:
column_names = [f"move_{i}" for i in range(X.shape[1])]
df_all = pd.DataFrame(X, columns=column_names)
df_all['target_move'] = Y
df_all['sample_weight'] = weight
df_all['game_id'] = game_id

In [22]:
unique_games = df_all['game_id'].unique()
np.random.seed(42)  
np.random.shuffle(unique_games)
train_games = round(len(unique_games)*0.8)
val_games = round(len(unique_games)*0.9)
ids_train = unique_games[:train_games]
ids_val = unique_games[val_games:]

In [23]:
df_train = df_all[df_all['game_id'].isin(ids_train)]
df_val = df_all[df_all['game_id'].isin(ids_val)]
df_test = df_all[(~df_all['game_id'].isin(ids_train)) & (~df_all['game_id'].isin(ids_val))]

In [24]:
df_train.shape

(1516245, 228)

In [25]:
df_val.shape

(193550, 228)

In [26]:
df_test.shape

(191874, 228)

In [27]:
df_train.shape[0]+df_val.shape[0]+df_test.shape[0]

1901669

In [53]:
model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=225,          
    n_estimators=300,       
    learning_rate=0.005,     
    num_leaves=225,          
    max_depth=9,
    min_child_samples=20,
    n_jobs=-1,             
    random_state=42
)

In [54]:
model.fit(
    df_train.iloc[:,:225], df_train['target_move'],
    sample_weight=df_train['sample_weight'],  
    eval_set=[(df_val.iloc[:,:225], df_val['target_move'])],
    eval_sample_weight=[df_val['sample_weight']],
    callbacks=[
        lgb.early_stopping(stopping_rounds=10, verbose=True)
    ]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.733076 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 674
[LightGBM] [Info] Number of data points in the train set: 4005535, number of used features: 225
[LightGBM] [Info] Start training from score -8.190961
[LightGBM] [Info] Start training from score -8.094691
[LightGBM] [Info] Start training from score -7.949500
[LightGBM] [Info] Start training from score -7.802215
[LightGBM] [Info] Start training from score -7.626954
[LightGBM] [Info] Start training from score -7.441397
[LightGBM] [Info] Start training from score -7.373524
[LightGBM] [Info] Start training from score -7.363845
[LightGBM] [Info] Start training from score -7.304293
[LightGBM] [Info] Start training from score -7.372235
[LightGBM] [Info] Start training from score -7.485131
[LightGBM] [Info] Start training from score -7.6084

LGBMClassifier(learning_rate=0.005, max_depth=9, n_estimators=300, n_jobs=-1,
               num_class=225, num_leaves=225, objective='multiclass',
               random_state=42)

In [13]:
model1 = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=225,          
    n_estimators=300,       
    learning_rate=0.01,     
    num_leaves=64,          
    max_depth=7,
    min_child_samples=20,
    n_jobs=-1,             
    random_state=42
)

In [14]:
model1.fit(
    df_train.iloc[:,:225], df_train['target_move'],
    sample_weight=df_train['sample_weight'],  
    eval_set=[(df_val.iloc[:,:225], df_val['target_move'])],
    eval_sample_weight=[df_val['sample_weight']],
    callbacks=[
        lgb.early_stopping(stopping_rounds=10, verbose=True)
    ]
)
model1.booster_.save_model('gomoku_lgb_champion_v1.txt')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.709432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 674
[LightGBM] [Info] Number of data points in the train set: 4005535, number of used features: 225
[LightGBM] [Info] Start training from score -8.190961
[LightGBM] [Info] Start training from score -8.094691
[LightGBM] [Info] Start training from score -7.949500
[LightGBM] [Info] Start training from score -7.802215
[LightGBM] [Info] Start training from score -7.626954
[LightGBM] [Info] Start training from score -7.441397
[LightGBM] [Info] Start training from score -7.373524
[LightGBM] [Info] Start training from score -7.363845
[LightGBM] [Info] Start training from score -7.304293
[LightGBM] [Info] Start training from score -7.372235
[LightGBM] [Info] Start training from score -7.485131
[LightGBM] [Info] Start training from score -7.6084

In [28]:
model2 = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=225,          
    n_estimators=300,       
    learning_rate=0.01,     
    num_leaves=128,          
    max_depth=8,
    min_child_samples=20,
    n_jobs=-1,             
    random_state=42
)

In [31]:
model2_v2 = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=225,          
    n_estimators=300,       
    learning_rate=0.005,     
    num_leaves=128,          
    max_depth=8,
    min_child_samples=20,
    n_jobs=-1,             
    random_state=42
)

In [13]:
model2.fit(
    df_train.iloc[:,:225], df_train['target_move'],
    sample_weight=df_train['sample_weight'],  
    eval_set=[(df_val.iloc[:,:225], df_val['target_move'])],
    eval_sample_weight=[df_val['sample_weight']],
    callbacks=[
        lgb.early_stopping(stopping_rounds=10, verbose=True)
    ]
)
model2.booster_.save_model('gomoku_lgb_champion_v2.txt')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.754860 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 674
[LightGBM] [Info] Number of data points in the train set: 4005535, number of used features: 225
[LightGBM] [Info] Start training from score -8.190961
[LightGBM] [Info] Start training from score -8.094691
[LightGBM] [Info] Start training from score -7.949500
[LightGBM] [Info] Start training from score -7.802215
[LightGBM] [Info] Start training from score -7.626954
[LightGBM] [Info] Start training from score -7.441397
[LightGBM] [Info] Start training from score -7.373524
[LightGBM] [Info] Start training from score -7.363845
[LightGBM] [Info] Start training from score -7.304293
[LightGBM] [Info] Start training from score -7.372235
[LightGBM] [Info] Start training from score -7.485131
[LightGBM] [Info] Start training from score -7.6084

In [32]:
model2_v2.fit(
    df_train.iloc[:,:225], df_train['target_move'],
    sample_weight=df_train['sample_weight'],  
    eval_set=[(df_val.iloc[:,:225], df_val['target_move'])],
    eval_sample_weight=[df_val['sample_weight']],
    callbacks=[
        lgb.early_stopping(stopping_rounds=10, verbose=True)
    ]
)
model2_v2.booster_.save_model('v2_gomoku_lgb_champion_v2.txt')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.300585 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 675
[LightGBM] [Info] Number of data points in the train set: 1516245, number of used features: 225
[LightGBM] [Info] Start training from score -8.211929
[LightGBM] [Info] Start training from score -8.733143
[LightGBM] [Info] Start training from score -8.587998
[LightGBM] [Info] Start training from score -8.028549
[LightGBM] [Info] Start training from score -7.194862
[LightGBM] [Info] Start training from score -7.600130
[LightGBM] [Info] Start training from score -7.967453
[LightGBM] [Info] Start training from score -7.826672
[LightGBM] [Info] Start training from score -7.662900
[LightGBM] [Info] Start training from score -7.400188
[LightGBM] [Info] Start training from score -7.823828
[LightGBM] [Info] Start training from score -7.5128

In [15]:
model3 = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=225,          
    n_estimators=300,       
    learning_rate=0.05,     
    num_leaves=225,          
    max_depth=9,
    min_child_samples=20,
    n_jobs=-1,             
    random_state=42
)

In [16]:
model3.fit(
    df_train.iloc[:,:225], df_train['target_move'],
    sample_weight=df_train['sample_weight'],  
    eval_set=[(df_val.iloc[:,:225], df_val['target_move'])],
    eval_sample_weight=[df_val['sample_weight']],
    callbacks=[
        lgb.early_stopping(stopping_rounds=10, verbose=True)
    ]
)
model1.booster_.save_model('gomoku_lgb_champion_v3.txt')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.699934 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 674
[LightGBM] [Info] Number of data points in the train set: 4005535, number of used features: 225
[LightGBM] [Info] Start training from score -8.190961
[LightGBM] [Info] Start training from score -8.094691
[LightGBM] [Info] Start training from score -7.949500
[LightGBM] [Info] Start training from score -7.802215
[LightGBM] [Info] Start training from score -7.626954
[LightGBM] [Info] Start training from score -7.441397
[LightGBM] [Info] Start training from score -7.373524
[LightGBM] [Info] Start training from score -7.363845
[LightGBM] [Info] Start training from score -7.304293
[LightGBM] [Info] Start training from score -7.372235
[LightGBM] [Info] Start training from score -7.485131
[LightGBM] [Info] Start training from score -7.6084

In [55]:

X_val = df_test.iloc[:,:225]
y_val = df_test['target_move']
y_pred = model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print(f"Ogólna dokładność (Accuracy): {acc:.2%}")

Ogólna dokładność (Accuracy): 47.11%


In [56]:
model.booster_.save_model('gomoku_lgb_champion.txt')

In [57]:
bst = lgb.Booster(model_file='gomoku_lgb_champion.txt')

In [ ]:
learning_rates = [0.01,0.1]
num_leaves = []